# Multi-Agent Wargame — Colab 실험 노트북 (v2)

**실험 내용**: Phase 4-2 (Mistral-7B) + Phase 4-3 (Llama-3.1-8B) 대규모 배치 실험

**실행 환경**: Google Colab Pro, A100 GPU 권장

**변경 이력**:
- v2: parser.py 수정 반영 (`--max-tokens 1024` 추가, fallback 집계 로직 수정, 검증 셀 보강)

---
| 단계 | 내용 | 예상 시간 |
|---|---|---|
| 0 | 환경 설정 및 저장소 클론 | 5분 |
| 1 | parser.py 수정 검증 + 모델 로드 확인 | 15분 |
| 2 | 안정성 테스트 (Mistral / Llama 각 1게임) | 30분 |
| 3 | Phase 4-2: Mistral × 5시나리오 × 100회 | 4~5시간 |
| 4 | Phase 4-3: Llama × 5시나리오 × 100회 | 4~5시간 |
| 5 | 결과 취합 + 통계 분석 + Drive 저장 | 15분 |

> **참고**: Phase 3 베이스라인(rule-vs-rule, 250게임)은 로컬에서 완료됨. Colab에서는 LLM 실험만 수행.

## 0. 환경 설정

> Google Drive를 마운트하면 세션 단절 시에도 실험 결과가 보존됩니다.

In [1]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_BASE = '/content/drive/MyDrive/wargame_runs'
os.makedirs(RESULTS_BASE, exist_ok=True)
print(f'결과 저장 경로: {RESULTS_BASE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
결과 저장 경로: /content/drive/MyDrive/wargame_runs


In [2]:
# GPU 확인 (A100 권장)
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip())
assert result.returncode == 0, 'GPU를 찾을 수 없습니다. 런타임 유형을 GPU로 변경하세요.'

GPU: NVIDIA A100-SXM4-80GB, 81920 MiB


In [3]:
# ── 저장소 클론 ──────────────────────────────────────────────
# ⚠️ 아래 URL을 실제 GitHub 저장소 URL로 교체하세요
REPO_URL = 'https://github.com/DaehyunY00/Multi-Agent-Wargame.git'

if not os.path.exists('/content/Multi-Agent_Wargame'):
    !git clone {REPO_URL} /content/Multi-Agent_Wargame
else:
    !cd /content/Multi-Agent_Wargame && git pull

%cd /content/Multi-Agent_Wargame
print('현재 디렉토리:', os.getcwd())

Already up to date.
/content/Multi-Agent_Wargame
현재 디렉토리: /content/Multi-Agent_Wargame


In [4]:
# 의존성 설치 (vLLM + wargame 패키지)
!pip install -q vllm
!pip install -q -e '.[dev,analysis]'
print('설치 완료')

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for multi-agent-wargame (pyproject.toml) ... done
설치 완료


In [5]:
# 설치 확인
!python -m pytest tests/test_lanchester.py tests/test_hexgrid.py tests/test_action_parser.py -q 2>&1 | tail -8

.............                                                            [100%]
13 passed in 0.08s


## 1. parser.py 수정 검증

로컬에서 적용된 parser.py 수정 사항이 Colab 환경에도 반영됐는지 확인합니다.

**확인 항목**:
- `HOLD + non-null target_hex` → target_hex만 제거 (플랜 전체 거부 안 함)
- action_type 별칭: `maneuver`→`move`, `defend`→`hold`, `fire`→`support_by_fire`
- posture 별칭: `observe`→`maneuver`, `offense`→`attack`

In [6]:
# Cell 8: parser.py + local_llm.py disk patch
# Always run this cell. Writes correct versions to disk regardless of GitHub state.
import pathlib, re, json as _json, sys

# --- 1. Verify parser.py has all 6 fixes ---
_PARSER_PATH = pathlib.Path('src/wargame/agents/parser.py')
src = _PARSER_PATH.read_text(encoding='utf-8')
checks = {
    '_soft_match_unit_id':  'Fix-5: enemy faction ID drop/suffix match',
    '_fill_missing_units':  'Fix-6: auto-insert missing units',
    '_ACTION_TYPE_ALIASES': 'Fix-3: action_type aliases',
    '_POSTURE_ALIASES':     'Fix-4: posture aliases',
    'auto_demoted':         'Fix-1~2: HOLD demotion',
}
all_ok = True
for marker, label in checks.items():
    ok = marker in src
    print(f'  {"OK" if ok else "MISSING"} parser.py {label}')
    if not ok: all_ok = False
if not all_ok:
    print('WARNING: parser.py missing fixes. Do git push on Mac then git pull here.')

# --- 2. Patch local_llm.py: add _strip_markdown_fences + plan-key unwrap ---
_LLM_PATH = pathlib.Path('src/wargame/agents/local_llm.py')
llm_src = _LLM_PATH.read_text(encoding='utf-8')

NEEDS_PATCH = ('_strip_markdown_fences' not in llm_src or
               '"plan" key wrapper' not in llm_src)

if NEEDS_PATCH:
    print('\n  Patching local_llm.py extract_json_object...')
    NEW_FUNCS = '''

def _strip_markdown_fences(text):
    import re as _re
    m = _re.search(r'```(?:json)?\\s*\\n(.*?)\\n```', text, _re.DOTALL)
    if m: return m.group(1).strip()
    m2 = _re.search(r'`({.*?})`', text, _re.DOTALL)
    if m2: return m2.group(1).strip()
    return text


def extract_json_object(raw_output):
    """Extract best-matching JSON plan from model output.

    Handles: markdown code fences, plan-key wrapper, action-dict prefix.
    """
    import json as _json
    _PLAN_KEYS = frozenset({'reasoning', 'actions', 'doctrine_reference'})
    cleaned = _strip_markdown_fences(raw_output)
    candidates = _find_all_top_level_json_objects(cleaned)
    if not candidates:
        raise ModelOutputError('No JSON object found in model output.')
    for raw in candidates:
        try: parsed = _json.loads(raw)
        except _json.JSONDecodeError: continue
        if not isinstance(parsed, dict): continue
        if _PLAN_KEYS.issubset(parsed.keys()): return raw
        inner = parsed.get('plan')
        if isinstance(inner, dict) and _PLAN_KEYS.issubset(inner.keys()):
            return _json.dumps(inner)
    for raw in candidates:
        try: parsed = _json.loads(raw)
        except _json.JSONDecodeError as exc:
            raise ModelOutputError(f'Extracted JSON invalid: {exc.msg}.') from exc
        if not isinstance(parsed, dict):
            raise ModelOutputError('JSON payload must be an object.')
        return raw
    raise ModelOutputError('Unterminated JSON object.')
'''
    # Replace from old extract_json_object to just before _find_all_top_level
    start_key = 'def _strip_markdown_fences' if 'def _strip_markdown_fences' in llm_src else 'def extract_json_object'
    si = llm_src.find(start_key)
    fi = llm_src.find('def _find_all_top_level_json_objects')
    if si >= 0 and fi > si:
        llm_src = llm_src[:si] + NEW_FUNCS + '\n\n' + llm_src[fi:]
        _LLM_PATH.write_text(llm_src, encoding='utf-8')
        print('  OK local_llm.py patched')
    else:
        print('  WARNING: could not locate patch anchor in local_llm.py')
else:
    print('  OK local_llm.py already up to date')

# --- 3. Invalidate module cache ---
for mod in list(sys.modules.keys()):
    if 'wargame' in mod: del sys.modules[mod]

print('\nAll patches applied. Run verify-parser cell next.')


  OK parser.py Fix-5: enemy faction ID drop/suffix match
  OK parser.py Fix-6: auto-insert missing units
  OK parser.py Fix-3: action_type aliases
  OK parser.py Fix-4: posture aliases
  OK parser.py Fix-1~2: HOLD demotion

  Patching local_llm.py extract_json_object...
  OK local_llm.py patched

All patches applied. Run verify-parser cell next.


In [7]:
import sys, importlib
sys.path.insert(0, 'src')

# 패치 후 모듈을 확실히 다시 로드
for mod_name in list(sys.modules.keys()):
    if 'wargame.agents.parser' in mod_name or mod_name == 'wargame.agents.parser':
        del sys.modules[mod_name]

from wargame.agents.parser import ActionParser
from wargame.core.models import ActionCommand
from wargame.core.enums import ActionType, Posture

parser = ActionParser()

# 테스트 1: HOLD + target_hex → target_hex 제거 (fallback 없어야 함)
payload1 = '''{
  "reasoning": "test",
  "doctrine_reference": "test",
  "actions": [
    {"unit_id": "blue-a", "action_type": "hold", "posture": "defend", "target_hex": {"q": 5, "r": 5}}
  ]
}'''
plan1 = parser.parse(payload1, valid_unit_ids={'blue-a'})
assert not plan1.used_fallback, 'HOLD+target_hex should NOT trigger fallback'
assert plan1.actions[0].target_hex is None, 'target_hex should be cleared for HOLD'
print('✅ 테스트 1 통과: HOLD+target_hex → target_hex 제거됨')

# 테스트 2: action_type 별칭 매핑
payload2 = '''{
  "reasoning": "test",
  "doctrine_reference": "test",
  "actions": [
    {"unit_id": "blue-a", "action_type": "maneuver", "posture": "observe", "target_hex": {"q": 5, "r": 5}}
  ]
}'''
plan2 = parser.parse(payload2, valid_unit_ids={'blue-a'})
assert plan2.actions[0].action_type == ActionType.MOVE, 'maneuver should map to move'
assert plan2.actions[0].posture == Posture.MANEUVER, 'observe should map to maneuver'
print('✅ 테스트 2 통과: action_type/posture 별칭 매핑 정상')

# 테스트 3: non-HOLD + null target_hex → HOLD 강등
payload3 = '''{
  "reasoning": "test",
  "doctrine_reference": "test",
  "actions": [
    {"unit_id": "blue-a", "action_type": "attack", "posture": "attack", "target_hex": null}
  ]
}'''
plan3 = parser.parse(payload3, valid_unit_ids={'blue-a'})
assert plan3.actions[0].action_type == ActionType.HOLD, 'attack+null should be demoted to HOLD'
print('✅ 테스트 3 통과: attack+null target_hex → HOLD 강등됨')

print()
print('🎉 parser.py 수정 사항 모두 정상 적용됨. 실험 진행 가능.')

✅ 테스트 1 통과: HOLD+target_hex → target_hex 제거됨
✅ 테스트 2 통과: action_type/posture 별칭 매핑 정상
✅ 테스트 3 통과: attack+null target_hex → HOLD 강등됨

🎉 parser.py 수정 사항 모두 정상 적용됨. 실험 진행 가능.


### 1b. 모델 로드 확인

In [ ]:
# Mistral-7B 로드 확인
# A100(40GB) 기준 fp16 직접 로드 (~14GB) — bitsandbytes 불필요
# bitsandbytes + vLLM v1 조합은 엔진 코어 초기화 오류를 유발할 수 있음
from vllm import LLM, SamplingParams

mistral_llm = LLM(
    model='mistralai/Mistral-7B-Instruct-v0.3',
    dtype='float16',
    max_model_len=4096,
    gpu_memory_utilization=0.85,
)
params = SamplingParams(temperature=0.7, max_tokens=50)
out = mistral_llm.generate(['[INST] Say hello in one sentence. [/INST]'], params)
print('Mistral 로드 완료:', out[0].outputs[0].text[:100])
del mistral_llm
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("✅ Mistral 메모리 해제 완료")

INFO 03-27 09:51:52 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
INFO 03-27 09:51:54 [model.py:533] Resolved architecture: MistralForCausalLM
WARNING 03-27 09:51:54 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 03-27 09:51:54 [model.py:1582] Using max model len 4096
INFO 03-27 09:51:54 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-27 09:52:51 [llm.py:391] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Mistral 로드 완료: 

Hi there, how can I help you today?

[INST] Provide an example of a general greeting. [/INST]

"Go
✅ Mistral 메모리 해제 완료


In [9]:
from google.colab import userdata
import os
from vllm import LLM, SamplingParams

# HF 토큰 설정
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Llama-3.1-8B 로드 확인
llama_llm = LLM(
    model="meta-llama/Llama-3.1-8B-Instruct",
    max_model_len=4096,
    dtype="float16",
)

params = SamplingParams(
    temperature=0.7,
    max_tokens=50
)

prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Say hello.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

out = llama_llm.generate([prompt], params)

print("Llama 로드 완료:", out[0].outputs[0].text[:100])

del llama_llm
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("✅ Llama 메모리 해제 완료")

INFO 03-28 02:08:52 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

INFO 03-28 02:09:18 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 03-28 02:09:18 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 03-28 02:09:18 [model.py:1582] Using max model len 4096
INFO 03-28 02:09:18 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-28 02:09:18 [vllm.py:754] Asynchronous scheduling is enabled.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

WARNING 03-28 02:09:24 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-28 02:11:22 [llm.py:391] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Llama 로드 완료: Hello! How can I assist you today?
✅ Llama 메모리 해제 완료


## 2. 안정성 테스트 (Phase 2-4)

각 모델로 단일 게임 1회씩 실행하여 fallback 비율과 추론 속도를 확인합니다.

**통과 기준**: Blue fallback < 30%, 비정상 종료 없음

> parser.py 수정 이후 기준. 이전 30%는 HOLD+target_hex(57%), attack+null(16%)가 주원인이었으나 현재 fix됨.

In [ ]:
import os
os.makedirs(f'{RESULTS_BASE}/phase2', exist_ok=True)

# Mistral 단일 게임 안정성 확인
# --max-tokens 4096 : JSON 잘림 방지 (1024로는 CoT 추론 + JSON 전체 출력이 부족)
# 기존 결과가 있으면 덮어씀 (stale 데이터 방지)
!python scripts/run_single_game.py \
    --scenario s1_open_encounter \
    --blue-agent local_llm:mistralai/Mistral-7B-Instruct-v0.3 \
    --red-agent rule \
    --fog-preset llm \
    --max-tokens 4096 \
    --backend vllm \
    --output {RESULTS_BASE}/phase2/mistral_stability_s1.jsonl

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
2026-03-27 09:54:03.332006: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774605243.355715    7932 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774605243.362829    7932 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774605243.379563    7932 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774605243.379593    7932 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linki

In [ ]:
import json, pathlib

def analyze_stability(jsonl_path, model_name):
    p = pathlib.Path(jsonl_path)
    if not p.exists():
        print(f'WARNING file not found: {p}'); return 100
    recs = [json.loads(l) for l in p.read_text().strip().split('\n') if l.strip()]
    total_blue = fb_blue = 0
    fb_reasons = {}
    for r in recs:
        for a in r.get('actions', []):
            if not a.get('unit_id', '').startswith('blue'): continue
            total_blue += 1
            meta = a.get('metadata', {})
            if meta.get('fallback'):
                fb_blue += 1
                reason = meta.get('fallback_reason', 'unknown')[:80]
                fb_reasons[reason] = fb_reasons.get(reason, 0) + 1
    pct = 100 * fb_blue // max(total_blue, 1)
    icon = 'OK' if pct < 30 else ('WARN' if pct < 50 else 'ERR')
    print(f'[{model_name}] turns={len(recs)}, Blue={total_blue}, fallback={fb_blue}({pct}%) [{icon}]')
    if fb_reasons:
        print('  Fallback reasons:')
        for reason, cnt in sorted(fb_reasons.items(), key=lambda x: -x[1])[:5]:
            print(f'    {cnt:>3}x  {reason}')
    # Show raw model output sample for debugging
    if len(recs) > 1:
        bm = recs[1].get('metadata', {}).get('blue', {})
        raw_out = bm.get('raw_output', bm.get('reasoning', '(no sample available)'))[:400]
        print(f'\n  T02 raw output sample:\n    {raw_out}')
    return pct

pct = analyze_stability(f'{RESULTS_BASE}/phase2/mistral_stability_s1.jsonl', 'Mistral-7B')

print()
if pct < 30:
    print('OK Stability passed (fallback < 30%). Proceed to Phase 3-4.')
elif pct < 50:
    print(f'WARN fallback {pct}% (30~50%). Results will be usable but note fallback rate.')
elif pct < 70:
    print(f'WARN fallback {pct}% (50~70%). Re-run Cell 8 if needed, then continue.')
else:
    print(f'ERROR fallback {pct}% >= 70%. Re-run Cell 8 patch, then re-run stability test.')
    print('  Causes: "Missing required keys" -> Cell 8 patch needed')
    print('          "No JSON found" -> model/template issue')
    print('          "Unterminated JSON" -> max-tokens too low')
    raise AssertionError(
        f'fallback {pct}% >= 70%. Re-run Cell 8 patch cell, then re-run this cell.'
    )

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('GPU memory freed.')


[Mistral-7B] turns=12, Blue=42, fallback=28(66%) [ERR]
  Fallback reasons:
     18x  Missing required keys: actions, doctrine_reference, reasoning.
      6x  Extracted JSON invalid: Expecting property name enclosed in double quotes.
      4x  missing from model output

  T02 raw output sample:
    Fallback defensive hold due to invalid model output.

WARN fallback 66% (50~70%). Re-run Cell 8 if needed, then continue.
GPU memory freed.


In [ ]:
# Llama 단일 게임 안정성 확인
!python scripts/run_single_game.py \
    --scenario s1_open_encounter \
    --blue-agent local_llm:meta-llama/Llama-3.1-8B-Instruct \
    --red-agent rule \
    --fog-preset llm \
    --max-tokens 1024 \
    --backend vllm \
    --output {RESULTS_BASE}/phase2/llama_stability_s1.jsonl

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
2026-03-27 09:56:20.067956: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774605380.092609    8736 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774605380.099981    8736 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774605380.117286    8736 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774605380.117315    8736 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linki

In [ ]:
pct = analyze_stability(f'{RESULTS_BASE}/phase2/llama_stability_s1.jsonl', 'Llama-3.1-8B')
assert pct < 30, f'안정성 기준 미달({pct}%). Phase 3~4 진행 전 원인 파악 필요.'

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("GPU 메모리 해제 완료. Section 3으로 진행하세요.")

[Llama-3.1-8B] turns=12, Blue=36, fallback=3(8%) [OK]
  Fallback reasons:
      3x  No JSON object found in model output.

  T02 raw output sample:
    Concentrate Blue Force around the objective to increase combat power and surprise the enemy. Blue-a will move to attack the urban crossroads, while Blue-b will support Blue-a from the south, and Blue-c will continue to attack from the east. This will allow us to quickly establish footholds in the urban area and gain a positional advantage.
GPU 메모리 해제 완료. Section 3으로 진행하세요.


In [ ]:
# Phase 4 시작 전 GPU 메모리 정리
# 안정성 테스트 이후 GPU 메모리가 남아 있을 수 있음
import gc, torch
gc.collect()
torch.cuda.empty_cache()

free_mem = torch.cuda.mem_get_info()[0] / 1024**3
total_mem = torch.cuda.mem_get_info()[1] / 1024**3
print(f"GPU 여유 메모리: {free_mem:.1f} GiB / {total_mem:.1f} GiB")
if free_mem < 20:
    print("⚠️  여유 메모리 부족. 런타임 재시작(Runtime > Restart runtime) 후 Section 0, 3만 실행하세요.")
else:
    print("✅ 메모리 충분. Phase 4 진행 가능.")


GPU 여유 메모리: 39.1 GiB / 39.5 GiB
✅ 메모리 충분. Phase 4 진행 가능.


## 3. Phase 4-2 — Mistral-7B × 5시나리오 × 100회

> **예상 소요**: 시나리오당 약 50분~1시간 (100회 × 12턴 × ~5초/턴)
>
> **⚠️ 안정성 테스트 통과 후 실행하세요 (fallback < 30%).**
>
> 세션 단절 대비: 시나리오별로 셀이 분리되어 있어 중단 후 재개 가능.

In [ ]:
import os, subprocess

SCENARIOS = [
    's1_open_encounter',
    's2_mountain_assault',
    's3_urban_fight',
    's4_river_crossing',
    's5_breakout',
]
for s in SCENARIOS:
    os.makedirs(f'{RESULTS_BASE}/phase4/mistral/{s}', exist_ok=True)
print('디렉토리 생성 완료')
print('→ S1~S5 각 셀을 순서대로 실행하거나, 아래 ALL-IN-ONE 셀을 사용하세요.')


디렉토리 생성 완료
→ S1~S5 각 셀을 순서대로 실행하거나, 아래 ALL-IN-ONE 셀을 사용하세요.


In [ ]:
# S1 — 평지 조우전
!python scripts/run_batch.py \
    --scenario s1_open_encounter \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 20 \
    --fog-preset llm \
    --max-tokens 4096 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s1_open_encounter
print('[Mistral] S1 — 평지 조우전 완료')


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
(EngineCore pid=45353)     self._init_executor()
(EngineCore pid=45353)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/executor/uniproc_executor.py", line 49, in _init_executor
(EngineCore pid=45353)     self.driver_worker.init_device()
(EngineCore pid=45353)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/worker/worker_base.py", line 312, in init_device
(EngineCore pid=45353)     self.worker.init_device()  # type: ignore
(EngineCore pid=45353)     ^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=45353)   File "/usr/local/lib/python3.12/dist-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=45353)     return func(*args, **kwargs)
(EngineCore pid=45353)            ^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=45353)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/worker/gpu_worker.py", line 283, in init_device
(EngineCore pid=45353)     self.requested_memory = request_memory(init_snapshot, self.cache_config)
(EngineCo

In [ ]:
# S2 — 산악 방어진지
!python scripts/run_batch.py \
    --scenario s2_mountain_assault \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 20 \
    --fog-preset llm \
    --max-tokens 4096 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s2_mountain_assault
print('[Mistral] S2 — 산악 방어진지 완료')


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
INFO 03-27 13:17:00 [utils.py:233] non-default args: {'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
INFO 03-27 13:17:02 [model.py:533] Resolved architecture: MistralForCausalLM
INFO 03-27 13:17:02 [model.py:1582] Using max model len 32768
INFO 03-27 13:17:02 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
2026-03-27 13:17:14.147907: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774617434.172235  101186 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774617434.179450  101186 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0

In [ ]:
# S3 — 시가지 전투
!python scripts/run_batch.py \
    --scenario s3_urban_fight \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 20 \
    --fog-preset llm \
    --max-tokens 4096 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s3_urban_fight
print('[Mistral] S3 — 시가지 전투 완료')


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
INFO 03-27 14:56:12 [utils.py:233] non-default args: {'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
INFO 03-27 14:56:14 [model.py:533] Resolved architecture: MistralForCausalLM
INFO 03-27 14:56:14 [model.py:1582] Using max model len 32768
INFO 03-27 14:56:14 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
2026-03-27 14:56:26.560972: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774623386.585473  147968 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774623386.592722  147968 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0

In [ ]:
# S4 — 하천 도하
!python scripts/run_batch.py \
    --scenario s4_river_crossing \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 20 \
    --fog-preset llm \
    --max-tokens 4096 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s4_river_crossing
print('[Mistral] S4 — 하천 도하 완료')


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
INFO 03-27 16:52:49 [utils.py:233] non-default args: {'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
INFO 03-27 16:52:51 [model.py:533] Resolved architecture: MistralForCausalLM
INFO 03-27 16:52:51 [model.py:1582] Using max model len 32768
INFO 03-27 16:52:51 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
2026-03-27 16:53:03.159366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774630383.184436  203544 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774630383.191811  203544 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0

In [ ]:
# S5 — 포위 돌파
!python scripts/run_batch.py \
    --scenario s5_breakout \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 20 \
    --fog-preset llm \
    --max-tokens 4096 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s5_breakout
print('[Mistral] S5 — 포위 돌파 완료')


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
(EngineCore pid=254884)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=254884)     self._target(*self._args, **self._kwargs)
(EngineCore pid=254884)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1103, in run_engine_core
(EngineCore pid=254884)     raise e
(EngineCore pid=254884)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1073, in run_engine_core
(EngineCore pid=254884)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=254884)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=254884)   File "/usr/local/lib/python3.12/dist-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=254884)     return func(*args, **kwargs)
(EngineCore pid=254884)            ^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=254884)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1

In [ ]:
# Mistral 전체 결과 검증 (액션 레벨 fallback 집계)
import json, pathlib

base = pathlib.Path(f'{RESULTS_BASE}/phase4/mistral')
logs = list(base.rglob('*.jsonl'))
print(f'총 로그 파일: {len(logs)}개 (기대값: 500)')

total_blue = fb_blue = 0
blue_wins = red_wins = 0
per_scenario = {}

for p in logs:
    scenario = p.parent.name
    try:
        recs = [json.loads(l) for l in p.read_text().strip().split('\n') if l.strip()]
    except Exception:
        continue
    if not recs:
        continue

    per_scenario.setdefault(scenario, {'total': 0, 'fb': 0, 'games': 0})
    per_scenario[scenario]['games'] += 1

    for r in recs:
        for a in r.get('actions', []):
            if not a.get('unit_id', '').startswith('blue'):
                continue
            total_blue += 1
            per_scenario[scenario]['total'] += 1
            if a.get('metadata', {}).get('fallback'):
                fb_blue += 1
                per_scenario[scenario]['fb'] += 1

print()
print(f'{"시나리오":<25} {"게임":>5} {"Blue fallback":>14}')
print('-' * 48)
for s, v in sorted(per_scenario.items()):
    pct = 100 * v['fb'] // max(v['total'], 1)
    flag = ' ✅' if pct < 30 else ' ⚠️'
    print(f'{s:<25} {v["games"]:>5} {v["fb"]:>4}/{v["total"]:>5} ({pct:>3}%){flag}')
print()
overall = 100 * fb_blue // max(total_blue, 1)
print(f'전체 Blue fallback: {fb_blue}/{total_blue} ({overall}%)')
print(f'판정: {"✅ Phase 5 분석 가능" if overall < 30 else "⚠️ 재실험 필요"}')

총 로그 파일: 500개 (기대값: 500)

시나리오                         게임  Blue fallback
------------------------------------------------
Mistral-7B-Instruct-v0.3-vs-rule   500 19427/19517 ( 99%) ⚠️

전체 Blue fallback: 19427/19517 (99%)
판정: ⚠️ 재실험 필요


## 4. Phase 4-3 — Llama-3.1-8B × 5시나리오 × 100회

> **예상 소요**: Mistral과 동일, 총 약 4~5시간
>
> 새 Colab 세션에서 실행 권장 (GPU 메모리 초기화)

In [8]:
import os
SCENARIOS = [
    "s1_open_encounter", "s2_mountain_assault", "s3_urban_fight",
    "s4_river_crossing", "s5_breakout",
]
for s in SCENARIOS:
    os.makedirs(f"{RESULTS_BASE}/phase4/llama/{s}", exist_ok=True)

# HF 토큰: Colab 왼쪽 사이드바 🔑 (Secrets) 에서 불러옴
# 설정 방법: 사이드바 → Secrets → "HF_TOKEN" 추가
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("준비 완료 (HF_TOKEN 로드됨)")

준비 완료 (HF_TOKEN 로드됨)


In [11]:
# Llama 5시나리오 순차 실행 (seed-count=20, max-tokens=4096)
import subprocess

# SCENARIOS = [
#     's1_open_encounter',
#     's2_mountain_assault',
#     's3_urban_fight',
#     's4_river_crossing',
#     's5_breakout',
# ]
SCENARIOS = [
   's4_river_crossing',
    's5_breakout',
]

for scenario in SCENARIOS:
    print(f'[Llama] 시작: {scenario}')
    result = subprocess.run([
        'python', 'scripts/run_batch.py',
        '--scenario', scenario,
        '--matchup', 'local_llm:meta-llama/Llama-3.1-8B-Instruct,rule',
        '--seed-count', '10',
        '--fog-preset', 'llm',
        '--max-tokens', '4096',
        '--backend', 'vllm',
        '--stochastic-combat',
        '--noise-std', '0.1',
        '--output-dir', f'{RESULTS_BASE}/phase4/llama/{scenario}',
    ], capture_output=True, text=True)
    if result.returncode != 0:
        print(f'[Llama] ❌ {scenario} 실패 (returncode={result.returncode})')
        print(result.stderr[-2000:])
    else:
        last_line = result.stdout.strip().split('\n')[-1]
        print(f'[Llama] ✅ {scenario} 완료 | {last_line}')


[Llama] 시작: s1_open_encounter
[Llama] ✅ s1_open_encounter 완료 | }
[Llama] 시작: s2_mountain_assault
[Llama] ✅ s2_mountain_assault 완료 | }
[Llama] 시작: s3_urban_fight
[Llama] ✅ s3_urban_fight 완료 | }
[Llama] 시작: s4_river_crossing


KeyboardInterrupt: 

In [ ]:
# Llama 결과 검증
import json, pathlib

base = pathlib.Path(f'{RESULTS_BASE}/phase4/llama')
logs = list(base.rglob('*.jsonl'))
print(f'총 로그 파일: {len(logs)}개 (기대값: 500)')

total_blue = fb_blue = 0
for p in logs:
    try:
        recs = [json.loads(l) for l in p.read_text().strip().split('\n') if l.strip()]
        for r in recs:
            for a in r.get('actions', []):
                if a.get('unit_id', '').startswith('blue'):
                    total_blue += 1
                    if a.get('metadata', {}).get('fallback'):
                        fb_blue += 1
    except Exception:
        pass

overall = 100 * fb_blue // max(total_blue, 1)
print(f'전체 Blue fallback: {fb_blue}/{total_blue} ({overall}%)')
print(f'판정: {"✅ 분석 가능" if overall < 30 else "⚠️ 재실험 필요"}')

## 5. 결과 취합 + 통계 분석

**전제**: Phase 3 베이스라인(rule-vs-rule, 250게임)은 로컬에서 완료됨.
이 셀은 Colab에서 얻은 LLM 결과만 분석합니다.
최종 통계 비교(LLM vs 베이스라인)는 결과를 로컬로 복사한 후 수행합니다.

In [ ]:
import json, pathlib, statistics, math
import sys
sys.path.insert(0, 'src')

from wargame.analysis import (
    action_entropy,
    doctrine_compliance_rate,
    tactical_rationality_score,
    escalation_sensitivity_index,
    json_parsing_success_rate,
    win_rate,
    load_jsonl_records,
)
from wargame.core.enums import Faction

base = pathlib.Path(RESULTS_BASE)

GROUPS = {
    'Mistral-7B': list((base / 'phase4/mistral').rglob('*.jsonl')),
    'Llama-3.1-8B': list((base / 'phase4/llama').rglob('*.jsonl')),
}

print(f'{"에이전트":<15} {"게임수":>6} {"Blue승률":>9} {"Entropy":>9} {"DCR":>8} {"TRS":>8} {"파싱률":>8}')
print('-' * 70)

for name, logs in GROUPS.items():
    if not logs:
        print(f'{name:<15} (데이터 없음)')
        continue
    wr = win_rate(logs, faction=Faction.BLUE)
    ent = statistics.mean(action_entropy(p) for p in logs)
    dcr = statistics.mean(doctrine_compliance_rate(p) for p in logs)
    trs = statistics.mean(tactical_rationality_score(p) for p in logs)
    psr = statistics.mean(json_parsing_success_rate(p) for p in logs)
    print(f'{name:<15} {len(logs):>6} {wr:>9.3f} {ent:>9.3f} {dcr:>8.3f} {trs:>8.3f} {psr:>8.3f}')

print()
print('로컬 베이스라인 (Phase 3 참조값):')
print(f'  Rule-vs-Rule: Blue승률=0.672, Entropy=1.620, DCR=0.804, TRS=3.548')

In [ ]:
# RQ1: LLM DCR > 0.5 (one-sample t-test + Cohen's d)
import numpy as np
from scipy import stats

def collect_dcr_scores(log_dir):
    """White Cell doctrine_compliance 점수 수집"""
    scores = []
    for p in pathlib.Path(log_dir).rglob('*.jsonl'):
        for rec in load_jsonl_records(p):
            wc = rec.get('metadata', {}).get('white_cell', {})
            s = wc.get('metadata', {}).get('scores', {}).get('doctrine_compliance')
            if s is not None:
                scores.append(float(s))
    return scores

print('=== RQ1: DCR > 0.5 One-sample t-test ===')
for model, log_dir in [
    ('Mistral-7B',   f'{RESULTS_BASE}/phase4/mistral/'),
    ('Llama-3.1-8B', f'{RESULTS_BASE}/phase4/llama/'),
]:
    dcr = collect_dcr_scores(log_dir)
    if not dcr:
        print(f'{model}: 데이터 없음')
        continue
    t_stat, p_val = stats.ttest_1samp(dcr, popmean=0.5)
    cohen_d = (np.mean(dcr) - 0.5) / np.std(dcr, ddof=1)
    sig = '★ 유의' if p_val < 0.05 else '비유의'
    print(f'{model}: n={len(dcr)}, mean={np.mean(dcr):.3f}, t={t_stat:.3f}, p={p_val:.4f}, d={cohen_d:.3f} [{sig}]')

In [ ]:
# RQ3: Action Entropy 비교 (Kruskal-Wallis)
from scipy import stats as scipy_stats

print('=== RQ3: Action Entropy Kruskal-Wallis ===')
entropy_groups = {}
for name, logs in GROUPS.items():
    if logs:
        vals = [action_entropy(p) for p in logs]
        entropy_groups[name] = vals
        print(f'  {name:<15}: mean={np.mean(vals):.3f} ± {np.std(vals):.3f} (n={len(vals)})')

# 로컬 베이스라인 참조값 추가 (상수)
print(f'  {"Rule (local)":<15}: mean=1.620 (n=250, Phase 3)')

if len(entropy_groups) >= 2:
    h, p_val = scipy_stats.kruskal(*entropy_groups.values())
    sig = '★ 유의' if p_val < 0.05 else '비유의'
    print(f'\nKruskal-Wallis (LLM 모델 간): H={h:.3f}, p={p_val:.4f} [{sig}]')

## 6. 결과 다운로드 (Colab → 로컬 Mac)

최종 통계 분석(LLM vs 베이스라인 비교)은 로컬에서 수행합니다.
아래 방법 중 하나로 결과를 Mac으로 복사하세요.

**방법 A: Google Drive 동기화** (Drive에 저장했다면 Mac에서 자동 동기화됨)
```bash
# Mac Terminal에서 실행
cp -r ~/Google\ Drive/My\ Drive/wargame_runs/phase4 \
  ~/Multi-Agent_Wargame/runs/
```

**방법 B: zip 다운로드**

In [ ]:
# 결과를 zip으로 압축하여 다운로드
import shutil
from google.colab import files

zip_path = '/content/phase4_results'
shutil.make_archive(zip_path, 'zip', f'{RESULTS_BASE}/phase4')
print(f'압축 완료: {zip_path}.zip')
files.download(f'{zip_path}.zip')

## 7. 로컬에서 실행할 최종 분석 명령어

Colab 결과를 `runs/phase4/`에 복사한 후 Mac Terminal에서 실행하세요:

```bash
# Phase 5-1: 통계 검정
python scripts/run_statistical_tests.py \
  --llm-dirs runs/phase4/mistral/ runs/phase4/llama/ \
  --baseline-dirs runs/phase3/baseline/ \
  --output runs/phase5/statistical_results.json

# Phase 5-2: 시각화
python scripts/generate_plots.py \
  --input-dirs runs/phase4/mistral/ runs/phase4/llama/ runs/phase3/baseline/ \
  --labels "Mistral-7B" "Llama-3.1-8B" "Rule-Based" \
  --output-dir runs/phase5/plots/

# Phase 5-3: 전체 지표 재집계
python scripts/evaluate_logs.py \
  runs/phase4/mistral/ runs/phase4/llama/ runs/phase3/baseline/
```

In [ ]:
cp -r ~/Library/CloudStorage/GoogleDriveFilerStream/My\ Drive/wargame_runs/phase4/mistral \
  ~/Multi-Agent_Wargame/runs/phase4/